### profiling experiments

In [61]:
import cProfile as cp
import functions_regular as fnr
import functions_subtrees as fns
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import importlib
import pstats
import test
import time
import json
from pstats import SortKey
importlib.reload(fnr)
importlib.reload(fns)

<module 'functions_subtrees' from '/Users/mbpr/Desktop/internship/Motif discovery/motif-discovery/src/functions_subtrees.py'>

## REGULAR VS SUBTREES

In [102]:
success_count = 0
total_count = 0

gTimes_3 = {}

for gSize in [25]:
    hTimes = {}
    hSize = 0
    delta_fnr = 0
    
    for hSize in range(10, 12, 1):
        
        samples_fnr = []
        samples_fns = []
        
        for m in range(10, 20):
            
            total_count += 1

            H = nx.scale_free_graph(hSize, seed=m)
            H = nx.to_undirected(H)
            J = nx.Graph()
            stringH = graphToString(H)
            J = parseStringToGraph(stringH)
            J.remove_edges_from(J.selfloop_edges())

            G = nx.scale_free_graph(gSize, seed=m+1)
            G = nx.to_undirected(G)
            K = nx.Graph()
            stringG = graphToString(G)
            K = parseStringToGraph(stringG)
            K.remove_edges_from(K.selfloop_edges())

            start_time_fnr = time.time()
            out_fnr = fnr.findSubgraphInstances(J, K, True)
            delta_fnr = time.time() - start_time_fnr

            stringG = graphToString(G)
            stringH = graphToString(H)
            
            start_time_fns = time.time()
            out_fns = fns.findSubgraphInstances(stringH, stringG)
            delta_fns = time.time() - start_time_fns
            
            if (out_fnr == out_fns):
                success_count += 1
                
                samples_fnr.append(delta_fnr)
                samples_fns.append(delta_fns)
        
        hTimes[hSize] = {'fnr': np.average(samples_fnr), 'fns': np.average(samples_fns)}
        
    gTimes_3[gSize] = hTimes
        
print(success_count/total_count)

0.65


## write to file

In [75]:
with open('profiling_tests/gTimes.json', 'w') as fp:
    json.dump(gTimes, fp)
    
with open('profiling_tests/gTimes_1.json', 'w') as fp:
    json.dump(gTimes_1, fp)

## read from file

In [ ]:
def js_r(filename):
   with open(filename) as f_in:
       return(json.load(f_in))

gTimes_1 = js_r('profiling_tests/gTimes_1.json')
gTimes = js_r('profiling_tests/gTimes.json')
print(gTimes_1)

In [96]:
print(gTimes_2)

{25: {8: {'fnr': 5.171756058931351, 'fns': 0.926334947347641}, 9: {'fnr': 2.7267900307973227, 'fns': 1.311586618423462}}}


In [105]:
results = {}
results = gTimes_1

results['25']['11'] = gTimes_3[25][11]


In [106]:
print(results)

{'15': {'1': {'fnr': 0.017673444747924805, 'fns': 0.01941354274749756}, '2': {'fnr': 0.016705012321472167, 'fns': 0.01918938159942627}, '3': {'fnr': 0.01748518943786621, 'fns': 0.020149588584899902}, '4': {'fnr': 0.04744291305541992, 'fns': 0.046487164497375486}, '5': {'fnr': 0.08409547805786133, 'fns': 0.07377724647521973}, '6': {'fnr': 0.1686870574951172, 'fns': 0.13082120418548585}, '7': {'fnr': 0.22242964638604057, 'fns': 0.20370446311102974}, '8': {'fnr': 0.5190408229827881, 'fns': 0.5074600279331207}, '9': {'fnr': 0.41852337973458426, 'fns': 0.5372131552015033}, '10': {'fnr': 1.134508106443617, 'fns': 1.723937537935045}, '11': {'fnr': 4.874379706382752, 'fns': 6.9850461959838865}, '12': {'fnr': 4.964514785342747, 'fns': 7.355334917704265}, '13': {'fnr': 35.42826144695282, 'fns': 57.234751629829404}, '14': {'fnr': 347.49004995822906, 'fns': 569.6131006002427}}, '25': {'1': {'fnr': 0.03075540065765381, 'fns': 0.03414771556854248}, '2': {'fnr': 0.030035066604614257, 'fns': 0.0329101

In [ ]:
from matplotlib.pyplot import figure
m = 0

G = nx.scale_free_graph(100, seed=m)
G = nx.to_undirected(G)

figure(num=None, figsize=(10, 10), dpi=80, facecolor='w', edgecolor='k')
plt.axis('off')
nx.draw_networkx(G, with_labels=False, node_size=5)

In [14]:
# input: edges of a graph written in bracketed pairs
# output: graph with corresponding edges
def parseStringToGraph(s):
    l = []

    if(s[len(s)-1] == ' '):
        s = s[:-1]

    if(s[0] == '[' and s[len(s)-1] == ']'):
        s = s[-(len(s)-1):-1]
        start = [key for key, val in enumerate(s) if val == '[']
        end = [key for key, val in enumerate(s) if val == ']']
        comma = [key for key, val in enumerate(s) if (val == ',' and not s[key-1] == ']')]

        for i in range(len(start)):
            left = [v for k, v in enumerate(s) if k in range(start[i]+1, comma[i])]
            right = [v for k, v in enumerate(s) if k in range(comma[i]+1, end[i])]

            left = int(''.join(left))
            right = int(''.join(right))
            l.append((left, right))

    H = nx.Graph()
    H.add_edges_from(l)

    return H


# reads the text file output by Josh and returns test name, hstring,
# gstring and answer in a dict
def parseFileToDict(file):
    f = open(file, 'r')
    D = {}
    index = 0
    first = True

    for line in f:
        if (line[0:4] == 'Test'):
            title = line[:-1]

        if (line[0] == '['):
            if (first):
                hstring = line[:-1]
                first = False
            else:
                gstring = line[:-1]
                first = True


        if (ord(line[0]) >= 48 and ord(line[0]) <= 57):
            ans = int(line)
            D[index] = {'title': title,
                        'hstring': hstring,
                        'gstring': gstring,
                        'answer': ans}
            index = index + 1

    return D


def test_findSubgraphInstances(file, which='regular'):

    D = parseFileToDict(file)

    for key in D.keys():
        print(D[key]['title'])
        H = nx.Graph()
        G = nx.Graph()

        hGraph = parseStringToGraph(D[key]['hstring'])
        gGraph = parseStringToGraph(D[key]['gstring'])

        H.add_edges_from(hGraph.edges())
        G.add_edges_from(gGraph.edges())

        if (which == 'regular'):
            count = fnr.findSubgraphInstances(H, G, True)

        if (which == 'subtrees'):
            count = fns.findSubgraphInstances(H, G, True)

        l = count
        ans = D[key]['answer']

        print('H: ',end='')
        print(D[key]['hstring'])
        print('G: ',end='')
        print(D[key]['gstring'])
#         print('instances: ',end='')
#         print([tuple(i.getMap()) for i in instances])
        print('correct number of instances: ',end='')
        print(ans)
        print('instances found: ',end='')
        print(l)

        if(l == ans):
            print('succeeded')
        else:
            print('failed')

        print()

                
# convert graphs to strings
def graphToString(G):
    edges = list(G.edges())
    string = '['
    
    for edge in edges:
        to_add = '[' + str(edge[0]) + ',' + str(edge[1]) + '], '
        string += to_add
        
    string = string[:len(string)-2]
    string += ']'
    
    return string